# 02 — Feature Engineering & DRI Construction
**Purpose:** Document the construction of the Displacement Risk Index (DRI) —
every weight, normalization choice, and threshold — with justification.

The DRI is a weighted composite of four normalized signals:

```
DRI = 0.35 × rent_burden_norm
    + 0.25 × rent_to_income_norm
    + 0.20 × low_vacancy_score
    + 0.20 × low_income_score
```

This notebook reproduces the logic in `src/features.py` transparently so that
any stakeholder can audit the methodology without reading code.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "iframe"


conn = sqlite3.connect("../housing_pulse.db")
df = pd.read_sql("SELECT * FROM census_tracts", conn)
conn.close()

print(f"Loaded {len(df):,} rows across {df['data_year'].nunique()} ACS years")

Loaded 2,012 rows across 2 ACS years


## 1. Weight Rationale

| Feature | Weight | Rationale |
|---|---|---|
| `rent_burden_pct` | **0.35** | Most direct measure of current financial stress. Households spending >50% of income on rent are at immediate displacement risk. This is the primary HUD affordability trigger. |
| `rent_to_income_ratio` | **0.25** | Forward indicator — captures affordability trajectory even when burden hasn't peaked yet. |
| `vacancy_rate` | **0.20** | Market tightness proxy. Low vacancy = fewer alternatives when pressure hits = less mobility. |
| `median_income` | **0.20** | Capacity to absorb increases. Low-income tracts have no buffer against rent shocks. |

The weights sum to 1.0. Rent burden leads because it is the most operationally
actionable signal — intervention programs (emergency rental assistance, tenant counseling)
target burdened households directly.

In [ ]:
# Step 1: low_vacancy_score — inverted vacancy rate
# High vacancy = more options for tenants = lower displacement pressure
df["low_vacancy_score"] = (1 - df["vacancy_rate"]).clip(0, 1)

# Step 2: low_income_score — normalized and inverted median income
# Higher income = more capacity to absorb rent increases = lower displacement pressure
income_max = df["median_income"].quantile(0.95)
income_min = df["median_income"].quantile(0.05)
df["low_income_score"] = (
    1 - (df["median_income"] - income_min) / (income_max - income_min)
).clip(0, 1)

# Step 3: rent_burden_norm — already a proportion, clip to [0,1]
df["rent_burden_norm"] = df["rent_burden_pct"].clip(0, 1)

# Step 4: rent_to_income_norm — normalize to 95th percentile
rti_max = df["rent_to_income_ratio"].quantile(0.95)
df["rti_norm"] = (df["rent_to_income_ratio"] / rti_max).clip(0, 1)

print("Component ranges:")
for col in ["rent_burden_norm","rti_norm","low_vacancy_score","low_income_score"]:
    print(f"  {col:25s}  min={df[col].min():.3f}  max={df[col].max():.3f}  mean={df[col].mean():.3f}")

Component ranges:
  rent_burden_norm           min=0.000  max=1.000  mean=0.247
  rti_norm                   min=0.176  max=1.000  mean=0.622
  low_vacancy_score          min=0.516  max=1.000  mean=0.921
  low_income_score           min=0.000  max=1.000  mean=0.632


## 2. Why Percentile Clipping Instead of Min-Max or Z-Score?

Z-score normalization is sensitive to outliers — a single extreme tract would
compress the entire distribution. Min-max normalization has the same problem.

Clipping at the 5th and 95th percentile before normalizing ensures that:
- The bulk of the distribution maps cleanly to [0, 1]
- Extreme outlier tracts still get a valid score (capped at 0 or 1)
- The normalization is robust to year-over-year data revisions

In [ ]:
# Compute DRI
weights = {"rent_burden_norm": 0.35, "rti_norm": 0.25,
           "low_vacancy_score": 0.20, "low_income_score": 0.20}
assert abs(sum(weights.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

df["displacement_risk_index"] = (
    df["rent_burden_norm"]  * weights["rent_burden_norm"] +
    df["rti_norm"]          * weights["rti_norm"] +
    df["low_vacancy_score"] * weights["low_vacancy_score"] +
    df["low_income_score"]  * weights["low_income_score"]
).round(4)

print(f"DRI range: {df['displacement_risk_index'].min():.3f} — {df['displacement_risk_index'].max():.3f}")
print(f"DRI mean:  {df['displacement_risk_index'].mean():.3f}")

DRI range: 0.227 — 0.875
DRI mean:  0.561


## 3. Tier Thresholds

| Range | Tier | Action |
|---|---|---|
| 0.00 – 0.30 | **Low** | Standard monitoring |
| 0.30 – 0.50 | **Moderate** | Quarterly review |
| 0.50 – 0.70 | **High** | Proactive outreach recommended |
| 0.70 – 1.00 | **Critical** | Immediate intervention required |

Thresholds were set to produce operationally meaningful group sizes —
small enough that Critical tracts can receive targeted intervention,
large enough that High tracts justify systematic outreach programs.

In [ ]:
df["risk_tier"] = pd.cut(
    df["displacement_risk_index"],
    bins=[0, 0.30, 0.50, 0.70, 1.0],
    labels=["Low","Moderate","High","Critical"],
    include_lowest=True
)

tier_dist = df.groupby(["data_year","risk_tier"], observed=True).size().unstack(fill_value=0)
print(tier_dist)

risk_tier  Low  Moderate  High  Critical
data_year                               
2022         7       277   503       100
2024         9       266   517       117


In [ ]:
fig = px.histogram(
    df.dropna(subset=["displacement_risk_index"]),
    x="displacement_risk_index", color="risk_tier", nbins=50,
    title="DRI Distribution by Risk Tier (Both Years)",
    color_discrete_map={"Low":"#2ecc71","Moderate":"#f39c12",
                        "High":"#e74c3c","Critical":"#8e44ad"},
    labels={"displacement_risk_index":"Displacement Risk Index"}
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()

## 4. Gentrification Pressure Flag
A tract is flagged when three conditions converge simultaneously:
- Median rent above the 60th percentile (elevated market demand)
- Median income below the 40th percentile (residents cannot absorb increases)
- Rent burden above 25% (pressure is already materializing)

All three must be true — this prevents false positives from high-rent
tracts that also have high incomes (e.g., Buckhead).

In [ ]:
df["gentrification_pressure_flag"] = (
    (df["median_rent"]    > df["median_rent"].quantile(0.60)) &
    (df["median_income"]  < df["median_income"].quantile(0.40)) &
    (df["rent_burden_pct"]> 0.25)
).astype(int)

total = len(df)
flagged = df["gentrification_pressure_flag"].sum()
print(f"Gentrification flags: {flagged} of {total} tracts ({flagged/total*100:.1f}%)")
print(df[df["gentrification_pressure_flag"]==1].groupby("county_name").size())

Gentrification flags: 45 of 2012 tracts (2.2%)
county_name
Cobb         3
DeKalb      15
Fulton       7
Gwinnett    20
dtype: int64


## 5. Sensitivity Check — What If We Change the Weights?

A legitimate concern with any composite index is whether results are sensitive
to the chosen weights. We test two alternative weight schemes:

- **Equal weights** (0.25 each) — no prior assumptions
- **Burden-heavy** (0.50 rent burden, rest split equally)

If the tier assignment is stable across schemes, the DRI is robust.

In [ ]:
def compute_dri(df, w_burden, w_rti, w_vacancy, w_income):
    return (
        df["rent_burden_norm"]  * w_burden  +
        df["rti_norm"]          * w_rti     +
        df["low_vacancy_score"] * w_vacancy +
        df["low_income_score"]  * w_income
    ).clip(0,1)

df["dri_equal"]  = compute_dri(df, 0.25, 0.25, 0.25, 0.25)
df["dri_burden"] = compute_dri(df, 0.50, 0.20, 0.15, 0.15)

# Spearman rank correlation — if high, tier ordering is stable
from scipy.stats import spearmanr
r_equal,  _ = spearmanr(df["displacement_risk_index"].dropna(), df["dri_equal"].dropna())
r_burden, _ = spearmanr(df["displacement_risk_index"].dropna(), df["dri_burden"].dropna())
print(f"Rank correlation vs equal weights:   {r_equal:.4f}")
print(f"Rank correlation vs burden-heavy:    {r_burden:.4f}")
print("Interpretation: >0.95 = tier ordering is robust to weight choice")

Rank correlation vs equal weights:   0.9887
Rank correlation vs burden-heavy:    0.9763
Interpretation: >0.95 = tier ordering is robust to weight choice
